# FTM Diff Model on NIPS 2017 Dataset

Use this notebook in Google Colab with GPU enabled. It runs `ftm_diff_model` on the 1000-image NIPS/ImageNet-compatible dataset and saves model-wise results.

## 1. Enable GPU

In Colab: `Runtime -> Change runtime type -> T4 GPU` or another GPU.

In [3]:
!nvidia-smi

Sun Aug 23 09:55:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Get Project Files

Option A: upload `ftm_diff_model` to Google Drive, then mount Drive and set `PROJECT_DIR`.

Option B: upload a zip of `ftm_diff_model` to Colab and unzip it.

In [9]:
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

In [4]:
from google.colab import drive
drive.mount('/content/drive')

# Change this path if your folder is somewhere else in Google Drive.
PROJECT_DIR = '/content/drive/MyDrive/ftm_diff_model'
%cd $PROJECT_DIR

Mounted at /content/drive
[Errno 2] No such file or directory: '/content/drive/MyDrive/ftm_diff_model'
/content


If you uploaded a zip directly to Colab instead of Drive, run this cell after uploading the zip in the Files panel.

In [5]:
# Optional zip workflow. Skip this if you mounted Google Drive above.
# !unzip -q /content/ftm_diff_model.zip -d /content
# %cd /content/ftm_diff_model

## 3. Install Dependencies

In [6]:
!pip install -q -r requirements.txt
!pip install -q -U timm

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


## 4. Verify NIPS Dataset

Both counts must be `1000`. This also verifies every CSV `ImageId` has a matching PNG file.

In [7]:
import csv, glob, os

image_files = glob.glob('data/images/*.png')
with open('data/images.csv', newline='') as f:
    rows = list(csv.DictReader(f))

image_ids = {os.path.splitext(os.path.basename(p))[0] for p in image_files}
missing = [r['ImageId'] for r in rows if r['ImageId'] not in image_ids]

print('images:', len(image_files))
print('csv rows:', len(rows))
print('missing:', len(missing))
if missing:
    print('first missing:', missing[:5])
assert len(image_files) == 1000
assert len(rows) == 1000
assert not missing

FileNotFoundError: [Errno 2] No such file or directory: 'data/images.csv'

## 5. Optional Debug Run

This attacks only `2 * batch_size` images. Run it first to confirm the environment works.

In [ ]:
!python main.py \
  --device cuda:0 \
  --batch_size 4 \
  --model_name ResNet50 \
  --save_dir ./exp/ResNet50/debug_nips \
  --eval \
  --debug

## 6. Full NIPS Run

This is the long run over all 1000 images. If Colab runs out of memory, reduce `--batch_size` to `10`, `5`, or `1`.

In [ ]:
!python main.py \
  --device cuda:0 \
  --batch_size 20 \
  --model_name ResNet50 \
  --save_dir ./exp/ResNet50/ftm_nips \
  --eval

## 7. Inspect Model-Wise Results

In [ ]:
import pandas as pd

modelwise_path = './exp/ResNet50/ftm_nips/results/model_wise_summary.csv'
details_path = './exp/ResNet50/ftm_nips/results/results_summary.csv'

modelwise = pd.read_csv(modelwise_path)
display(modelwise)
print('Detailed per-image/per-model results:', details_path)

## 8. Zip Results

In [ ]:
!zip -r ftm_nips_results.zip exp/ResNet50/ftm_nips

If running from Drive, the zip is saved in the same Drive folder. If running from `/content`, download `ftm_nips_results.zip` from the Colab Files panel.